# Fundamentos de Ingeniería de Datos — Actividad 3
## MODELADO: de la capa TRANSFORMED a un esquema estrella (capa SERVING)

**Equipo:** Bethzy Sarahi Salcido Toro (220443324) · Guillermo Fernando Ornelas Jimenez (225361237)

Evidencia **E3.1 — Diseño de esquema**. Unidad 3: 3.1 Diseño lógico y físico en SQL · 3.2 Modelos analíticos (star schema, hechos y dimensiones).

```
RAW (SQLite)        EXTRACTED (SQLite)          TRANSFORMED (Parquet)                 SERVING (hoy)
data            →   extracted_<etiqueta>    →   transformed_<etiqueta>.parquet    →   Modelado_de_Datos/serving.db (DuckDB)
bronze              bronze                      silver                                gold · 3 esquemas estrella
```

| Capa | Dónde vive | Qué contiene | Medallion |
|---|---|---|---|
| RAW / EXTRACTED | `bases/tabla_N.db` (SQLite) | Datos como llegaron + metadatos de extracción | Bronze |
| TRANSFORMED | `Transformacion_de_Datos/Processed/transformed_<etiqueta>.parquet` | Datos limpios, una tabla plana | Silver |
| SERVING | `Modelado_de_Datos/serving.db` (DuckDB) | Esquema estrella: hechos + dimensiones | Gold |

Se trabajan los **3 datasets** (`adilshamim8`, `samithsachidanandan`, `saurabhbadole`). Como comparten el mismo esquema de 22 columnas, el diseño se escribe **una sola vez** en la función `modelar(etiqueta)`, igual que `extraer(etiqueta)` y `transformar(etiqueta)` en la Actividad 2. Los tres modelos viven en el mismo `serving.db`; para que **ningún nombre de tabla se repita**, cada tabla lleva la etiqueta del dataset como sufijo (`fact_salarios_adilshamim8`, `dim_puesto_saurabhbadole`, …).

### Configuración del entorno

In [1]:
# ── Importar tools.py (nuestro archivo de utilidades) ─────────────────────
import sys, os
sys.path.append(os.path.expanduser('~/Documents/IA/4_semestre/ing_de_datos/ing_datos/tools'))
import tools

import duckdb
import pandas as pd
from IPython.display import display

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 200)

M = tools.modelado
os.makedirs(tools.RUTA_MODELADO, exist_ok=True)

# Si serving.db no existe, DuckDB lo crea
con = duckdb.connect(str(tools.SERVING_DB))

print(f'DuckDB {duckdb.__version__}')
print('Datasets:', tools.ETIQUETAS)
for et in tools.ETIQUETAS:
    print(f'  Origen  : {M.parquet(et).relative_to(tools.BASE_DIR)}')
print(f'  Destino : {tools.SERVING_DB.relative_to(tools.BASE_DIR)}')

DuckDB 1.5.6
Datasets: ['adilshamim8', 'samithsachidanandan', 'saurabhbadole']
  Origen  : Transformacion_de_Datos/Processed/transformed_adilshamim8.parquet
  Origen  : Transformacion_de_Datos/Processed/transformed_samithsachidanandan.parquet
  Origen  : Transformacion_de_Datos/Processed/transformed_saurabhbadole.parquet
  Destino : Modelado_de_Datos/serving.db


---
## PASO 1 — Conocer la tabla plana

Una vista temporal (`TEMP VIEW`) por dataset: guarda solo la consulta, no los datos. Cada vez que usamos `base_<etiqueta>`, DuckDB lee el Parquet. Es temporal para que `serving.db` no guarde la ruta de un archivo que solo existe en una computadora.

In [2]:
for et in tools.ETIQUETAS:
    vista = M.crear_vista(con, et)
    filas = con.sql(f'SELECT COUNT(*) FROM {vista}').fetchall()[0][0]
    print(f'{vista:<28} {filas:>8,} filas')

con.sql('SELECT * FROM base_adilshamim8 LIMIT 5').df()

base_adilshamim8               71,913 filas
base_samithsachidanandan       44,244 filas
base_saurabhbadole             46,637 filas


,work_year,experience_level,employment_type,job_title,salary,salary_currency,salary_in_usd,employee_residence,remote_ratio,company_location,company_size,_batch_id,_extracted_at,_source_table,_row_hash,es_remoto_total,misma_ubicacion,pago_en_usd,tasa_cambio_a_usd,_outlier_salario,_transformed_at,_transform_version
0,2025,EX,FT,Head of Data,348516,USD,348516,US,0,US,M,1,2026-11-07 22:43:08,data,95b566db6a71c35f,False,True,True,1.0,True,2026-11-07 22:44:35,v1.0
1,2025,EX,FT,Head of Data,232344,USD,232344,US,0,US,M,1,2026-11-07 22:43:08,data,d3a6b1cd671b1e99,False,True,True,1.0,False,2026-11-07 22:44:35,v1.0
2,2025,SE,FT,Data Scientist,145400,USD,145400,US,0,US,M,1,2026-11-07 22:43:08,data,7150c9650f025496,False,True,True,1.0,False,2026-11-07 22:44:35,v1.0
3,2025,SE,FT,Data Scientist,81600,USD,81600,US,0,US,M,1,2026-11-07 22:43:08,data,4ff664fae1d58f98,False,True,True,1.0,False,2026-11-07 22:44:35,v1.0
4,2025,MI,FT,Engineer,160000,USD,160000,US,100,US,M,1,2026-11-07 22:43:08,data,ecf9cbea6e0b1327,True,True,True,1.0,False,2026-11-07 22:44:35,v1.0


In [3]:
# ── Columnas y tipos (los tres Parquet tienen el mismo esquema) ────────────
con.sql('DESCRIBE base_adilshamim8').df()[['column_name', 'column_type']]

,column_name,column_type
0,work_year,SMALLINT
1,experience_level,VARCHAR
2,employment_type,VARCHAR
3,job_title,VARCHAR
4,salary,BIGINT
5,salary_currency,VARCHAR
6,salary_in_usd,BIGINT
7,employee_residence,VARCHAR
8,remote_ratio,TINYINT
9,company_location,VARCHAR


---
## PASO 2 — Diseño lógico: ¿qué tablas necesitamos? (3.1)

La tabla plana repite el mismo texto en muchas filas: el mismo puesto, el mismo país o el mismo nivel aparecen miles de veces. Si un código cambiara (por ejemplo, decodificar `SE` como *Senior*), habría que corregir miles de filas (**anomalía de actualización**). Normalizar es separar lo que se repite en su propia tabla para guardarlo una sola vez.

In [4]:
# ── ¿Cuánto se repite cada columna? ────────────────────────────────────────
COLS_ANALISIS = ['work_year', 'experience_level', 'employment_type', 'job_title', 'salary',
                 'salary_currency', 'salary_in_usd', 'employee_residence', 'remote_ratio',
                 'company_location', 'company_size', 'es_remoto_total', 'misma_ubicacion',
                 'pago_en_usd', 'tasa_cambio_a_usd', '_outlier_salario', '_row_hash']

repeticiones = {}
for et in tools.ETIQUETAS:
    repeticiones[et] = M.repeticiones(con, f'base_{et}', COLS_ANALISIS)

tabla_rep = pd.concat({et: df.set_index('columna')['valores_distintos'] for et, df in repeticiones.items()}, axis=1)
tabla_rep

,adilshamim8,samithsachidanandan,saurabhbadole
columna,,,
filas,71913,44244,46637
work_year,6,6,6
experience_level,4,4,4
employment_type,4,4,4
job_title,422,312,317
salary,12308,8331,8673
salary_currency,26,26,26
salary_in_usd,13580,9164,9535
employee_residence,104,96,96


In [5]:
# ── Detalle de un dataset: % de distintos y cuántas veces se repite cada valor ──
repeticiones['adilshamim8']

,columna,valores_distintos,pct_de_filas,veces_que_se_repite_en_promedio
0,filas,71913,100.00,1.0
1,work_year,6,0.01,11985.5
2,experience_level,4,0.01,17978.2
3,employment_type,4,0.01,17978.2
4,job_title,422,0.59,170.4
5,salary,12308,17.12,5.8
6,salary_currency,26,0.04,2765.9
7,salary_in_usd,13580,18.88,5.3
8,employee_residence,104,0.14,691.5
9,remote_ratio,3,0.00,23971.0


**Decisión de diseño.** Hay decenas de miles de filas, pero solo 6 años, 4 niveles, 4 tipos de empleo, 3 tamaños, 3 modalidades, 26 monedas, ~100 países y ~300–420 puestos. Cada columna que se repite es una **dimensión**. Los números que se suman o promedian (`salary`, `salary_in_usd`, `tasa_cambio_a_usd`) son los **hechos**. `_row_hash` es distinto en cada fila: no se repite, así que viaja en la tabla de hechos como llave de *lineage*.

| Columna(s) de la tabla plana (`transformed_<etiqueta>.parquet`) | Se vuelve… | Tipo |
|---|---|---|
| `work_year` | `dim_tiempo_<etiqueta>` | Dimensión |
| `experience_level` | `dim_experiencia_<etiqueta>` | Dimensión |
| `employment_type` | `dim_empleo_<etiqueta>` | Dimensión |
| `job_title` | `dim_puesto_<etiqueta>` | Dimensión |
| `employee_residence`, `company_location` | `dim_pais_<etiqueta>` (una sola tabla de países, usada dos veces: *role-playing*) | Dimensión |
| `company_size` | `dim_tamano_empresa_<etiqueta>` | Dimensión |
| `salary_currency`, `pago_en_usd` | `dim_moneda_<etiqueta>` (`pago_en_usd` depende solo de la moneda → `es_usd`) | Dimensión |
| `remote_ratio`, `es_remoto_total` | `dim_modalidad_<etiqueta>` (`es_remoto_total` depende solo de `remote_ratio`) | Dimensión |
| `salary`, `salary_in_usd`, `tasa_cambio_a_usd` | `fact_salarios_<etiqueta>` → `salario`, `salario_usd`, `tasa_cambio_a_usd` | **Medidas** |
| `misma_ubicacion`, `_outlier_salario` | `fact_salarios_<etiqueta>` (dependen de la fila, no de una sola dimensión) | Banderas en hechos |
| `_row_hash`, `_batch_id` | `fact_salarios_<etiqueta>` | Lineage |
| `_extracted_at`, `_source_table`, `_transformed_at`, `_transform_version` | No se copian: tienen un solo valor por dataset y ya están en el lineage JSON de la Actividad 2 | — |

---
## PASO 3 — Construir el esquema estrella (3.1 físico + 3.2)

```
                        dim_tiempo        dim_experiencia
                              \                /
         dim_puesto ──────  fact_salarios_<etiqueta>  ────── dim_empleo
                           /     |      |      \
           dim_pais (residencia) | dim_moneda   dim_modalidad
           dim_pais (empresa) ───┘      |
                                 dim_tamano_empresa
```

**Grano:** una fila de `fact_salarios_<etiqueta>` = **un registro salarial** (un puesto, en un año, con su salario), igual que una fila de la tabla plana.

| Elemento del diseño físico | Para qué |
|---|---|
| Tipos (`SMALLINT`, `TINYINT`, `BIGINT`, `DOUBLE`, `VARCHAR`, `BOOLEAN`) | Se conservan los tipos que dejó la Actividad 2 (`work_year` ya es entero) |
| `PRIMARY KEY` | Identifica cada fila; no se repite |
| `ROW_NUMBER()` | Genera la llave numérica (sustituta) de las dimensiones de texto |
| Llave natural | `dim_tiempo` usa el año y `dim_modalidad` el propio `remote_ratio` (0, 50, 100) |
| `REFERENCES` | Llave foránea: cada hecho apunta a una dimensión que sí existe |
| Catálogos (`VALUES`) | Decodifican los códigos (`SE` → Senior, `FT` → Tiempo completo), algo que en la Actividad 2 quedó pendiente |

In [6]:
# ── Catálogos para decodificar los códigos de la fuente ────────────────────
CAT_EXPERIENCIA = "VALUES ('EN', 'Inicial (Entry/Junior)', 1), ('MI', 'Intermedio (Mid)', 2), ('SE', 'Senior', 3), ('EX', 'Ejecutivo / Director', 4)"
CAT_EMPLEO      = "VALUES ('FT', 'Tiempo completo'), ('PT', 'Medio tiempo'), ('CT', 'Contrato'), ('FL', 'Freelance')"
CAT_TAMANO      = "VALUES ('S', 'Pequeña (< 50 empleados)', 1), ('M', 'Mediana (50 a 250)', 2), ('L', 'Grande (> 250)', 3)"
CAT_MODALIDAD   = "VALUES (0, 'Presencial'), (50, 'Híbrido'), (100, 'Remoto total')"


def tablas_del_modelo(et):
    """Nombres de las tablas de un dataset. Los hechos van primero para poder borrarlos antes."""
    return [f'fact_salarios_{et}', f'dim_tiempo_{et}', f'dim_experiencia_{et}', f'dim_empleo_{et}',
            f'dim_puesto_{et}', f'dim_pais_{et}', f'dim_tamano_empresa_{et}', f'dim_moneda_{et}',
            f'dim_modalidad_{et}']


def relaciones(et):
    """(columna FK en hechos, dimensión, columna PK) — para la prueba de huérfanas."""
    return [('anio', f'dim_tiempo_{et}', 'anio'),
            ('experiencia_id', f'dim_experiencia_{et}', 'experiencia_id'),
            ('empleo_id', f'dim_empleo_{et}', 'empleo_id'),
            ('puesto_id', f'dim_puesto_{et}', 'puesto_id'),
            ('residencia_pais_id', f'dim_pais_{et}', 'pais_id'),
            ('empresa_pais_id', f'dim_pais_{et}', 'pais_id'),
            ('tamano_id', f'dim_tamano_empresa_{et}', 'tamano_id'),
            ('moneda_id', f'dim_moneda_{et}', 'moneda_id'),
            ('remote_ratio', f'dim_modalidad_{et}', 'remote_ratio')]

In [7]:
def modelar(et):
    """Construye el esquema estrella de un dataset en serving.db a partir de base_<et>."""
    b = f'base_{et}'

    # ── Empezar de cero: primero los hechos, porque apuntan a las dimensiones ──
    M.borrar_tablas(con, tablas_del_modelo(et))

    # ── dim_tiempo: la llave es el propio año ──────────────────────────────
    con.execute(f"""
        CREATE TABLE dim_tiempo_{et} (
            anio                  SMALLINT PRIMARY KEY,
            decada                SMALLINT NOT NULL,
            es_anio_mas_reciente  BOOLEAN  NOT NULL
        )""")
    con.execute(f"""
        INSERT INTO dim_tiempo_{et}
        SELECT DISTINCT work_year,
               work_year // 10 * 10,
               work_year = (SELECT MAX(work_year) FROM {b})
        FROM {b}""")

    # ── dim_experiencia: código + descripción + orden (para ordenar EN < MI < SE < EX) ──
    con.execute(f"""
        CREATE TABLE dim_experiencia_{et} (
            experiencia_id  INTEGER PRIMARY KEY,
            codigo          VARCHAR NOT NULL UNIQUE,
            nivel           VARCHAR NOT NULL,
            orden           INTEGER NOT NULL
        )""")
    con.execute(f"""
        INSERT INTO dim_experiencia_{et}
        SELECT ROW_NUMBER() OVER (ORDER BY COALESCE(c.orden, 99), d.codigo),
               d.codigo, COALESCE(c.nivel, 'Desconocido'), COALESCE(c.orden, 99)
        FROM (SELECT DISTINCT experience_level AS codigo FROM {b}) AS d
        LEFT JOIN ({CAT_EXPERIENCIA}) AS c(codigo, nivel, orden) USING (codigo)""")

    # ── dim_empleo ──────────────────────────────────────────────────────────
    con.execute(f"""
        CREATE TABLE dim_empleo_{et} (
            empleo_id   INTEGER PRIMARY KEY,
            codigo      VARCHAR NOT NULL UNIQUE,
            tipo        VARCHAR NOT NULL
        )""")
    con.execute(f"""
        INSERT INTO dim_empleo_{et}
        SELECT ROW_NUMBER() OVER (ORDER BY d.codigo), d.codigo, COALESCE(c.tipo, 'Desconocido')
        FROM (SELECT DISTINCT employment_type AS codigo FROM {b}) AS d
        LEFT JOIN ({CAT_EMPLEO}) AS c(codigo, tipo) USING (codigo)""")

    # ── dim_puesto ──────────────────────────────────────────────────────────
    con.execute(f"""
        CREATE TABLE dim_puesto_{et} (
            puesto_id   INTEGER PRIMARY KEY,
            puesto      VARCHAR NOT NULL UNIQUE
        )""")
    con.execute(f"""
        INSERT INTO dim_puesto_{et}
        SELECT ROW_NUMBER() OVER (ORDER BY job_title), job_title
        FROM (SELECT DISTINCT job_title FROM {b})""")

    # ── dim_pais: una sola tabla para la residencia del empleado y la ubicación de la empresa ──
    con.execute(f"""
        CREATE TABLE dim_pais_{et} (
            pais_id      INTEGER PRIMARY KEY,
            codigo_iso2  VARCHAR NOT NULL UNIQUE
        )""")
    con.execute(f"""
        INSERT INTO dim_pais_{et}
        SELECT ROW_NUMBER() OVER (ORDER BY codigo), codigo
        FROM (SELECT employee_residence AS codigo FROM {b}
              UNION
              SELECT company_location FROM {b})""")

    # ── dim_tamano_empresa ──────────────────────────────────────────────────
    con.execute(f"""
        CREATE TABLE dim_tamano_empresa_{et} (
            tamano_id   INTEGER PRIMARY KEY,
            codigo      VARCHAR NOT NULL UNIQUE,
            tamano      VARCHAR NOT NULL
        )""")
    con.execute(f"""
        INSERT INTO dim_tamano_empresa_{et}
        SELECT ROW_NUMBER() OVER (ORDER BY COALESCE(c.orden, 99), d.codigo), d.codigo, COALESCE(c.tamano, 'Desconocido')
        FROM (SELECT DISTINCT company_size AS codigo FROM {b}) AS d
        LEFT JOIN ({CAT_TAMANO}) AS c(codigo, tamano, orden) USING (codigo)""")

    # ── dim_moneda: es_usd sale de la moneda (antes pago_en_usd en la tabla plana) ──
    con.execute(f"""
        CREATE TABLE dim_moneda_{et} (
            moneda_id   INTEGER PRIMARY KEY,
            codigo      VARCHAR NOT NULL UNIQUE,
            es_usd      BOOLEAN NOT NULL
        )""")
    con.execute(f"""
        INSERT INTO dim_moneda_{et}
        SELECT ROW_NUMBER() OVER (ORDER BY salary_currency), salary_currency, salary_currency = 'USD'
        FROM (SELECT DISTINCT salary_currency FROM {b})""")

    # ── dim_modalidad: la llave es el propio remote_ratio (0, 50, 100) ─────
    con.execute(f"""
        CREATE TABLE dim_modalidad_{et} (
            remote_ratio      TINYINT PRIMARY KEY,
            modalidad         VARCHAR NOT NULL,
            es_remoto_total   BOOLEAN NOT NULL
        )""")
    con.execute(f"""
        INSERT INTO dim_modalidad_{et}
        SELECT d.remote_ratio, COALESCE(c.modalidad, 'Desconocido'), d.remote_ratio = 100
        FROM (SELECT DISTINCT remote_ratio FROM {b}) AS d
        LEFT JOIN ({CAT_MODALIDAD}) AS c(remote_ratio, modalidad) USING (remote_ratio)""")

    # ── fact_salarios: el centro de la estrella (solo llaves, medidas y lineage) ──
    con.execute(f"""
        CREATE TABLE fact_salarios_{et} (
            salario_id          INTEGER PRIMARY KEY,

            anio                SMALLINT NOT NULL REFERENCES dim_tiempo_{et} (anio),
            experiencia_id      INTEGER  NOT NULL REFERENCES dim_experiencia_{et} (experiencia_id),
            empleo_id           INTEGER  NOT NULL REFERENCES dim_empleo_{et} (empleo_id),
            puesto_id           INTEGER  NOT NULL REFERENCES dim_puesto_{et} (puesto_id),
            residencia_pais_id  INTEGER  NOT NULL REFERENCES dim_pais_{et} (pais_id),
            empresa_pais_id     INTEGER  NOT NULL REFERENCES dim_pais_{et} (pais_id),
            tamano_id           INTEGER  NOT NULL REFERENCES dim_tamano_empresa_{et} (tamano_id),
            moneda_id           INTEGER  NOT NULL REFERENCES dim_moneda_{et} (moneda_id),
            remote_ratio        TINYINT  NOT NULL REFERENCES dim_modalidad_{et} (remote_ratio),

            salario             BIGINT   NOT NULL,
            salario_usd         BIGINT   NOT NULL,
            tasa_cambio_a_usd   DOUBLE,
            misma_ubicacion     BOOLEAN,
            es_atipico          BOOLEAN,

            _row_hash           VARCHAR  NOT NULL UNIQUE,
            _batch_id           INTEGER
        )""")
    con.execute(f"""
        INSERT INTO fact_salarios_{et}
        SELECT ROW_NUMBER() OVER (ORDER BY b._row_hash)  AS salario_id,
               b.work_year,
               x.experiencia_id, e.empleo_id, p.puesto_id,
               pr.pais_id, pe.pais_id,
               t.tamano_id, m.moneda_id, b.remote_ratio,
               b.salary, b.salary_in_usd, b.tasa_cambio_a_usd,
               b.misma_ubicacion, b._outlier_salario,
               b._row_hash, b._batch_id
        FROM {b} AS b
        JOIN dim_experiencia_{et}    AS x  ON x.codigo  = b.experience_level
        JOIN dim_empleo_{et}         AS e  ON e.codigo  = b.employment_type
        JOIN dim_puesto_{et}         AS p  ON p.puesto  = b.job_title
        JOIN dim_pais_{et}           AS pr ON pr.codigo_iso2 = b.employee_residence
        JOIN dim_pais_{et}           AS pe ON pe.codigo_iso2 = b.company_location
        JOIN dim_tamano_empresa_{et} AS t  ON t.codigo  = b.company_size
        JOIN dim_moneda_{et}         AS m  ON m.codigo  = b.salary_currency""")

    filas = {t: con.sql(f'SELECT COUNT(*) FROM {t}').fetchall()[0][0] for t in tablas_del_modelo(et)}
    print(f'[{et}] ' + ' | '.join(f'{t.replace("_" + et, "")}={n:,}' for t, n in filas.items()))
    return filas

In [8]:
resumen_modelo = {et: modelar(et) for et in tools.ETIQUETAS}

[adilshamim8] fact_salarios=71,913 | dim_tiempo=6 | dim_experiencia=4 | dim_empleo=4 | dim_puesto=422 | dim_pais=106 | dim_tamano_empresa=3 | dim_moneda=26 | dim_modalidad=3


[samithsachidanandan] fact_salarios=44,244 | dim_tiempo=6 | dim_experiencia=4 | dim_empleo=4 | dim_puesto=312 | dim_pais=99 | dim_tamano_empresa=3 | dim_moneda=26 | dim_modalidad=3


[saurabhbadole] fact_salarios=46,637 | dim_tiempo=6 | dim_experiencia=4 | dim_empleo=4 | dim_puesto=317 | dim_pais=99 | dim_tamano_empresa=3 | dim_moneda=26 | dim_modalidad=3


In [9]:
# ── Resultado: los tres esquemas estrella en serving.db ────────────────────
filas_modelo = pd.DataFrame({et: {t.replace('_' + et, ''): n for t, n in r.items()} for et, r in resumen_modelo.items()})
filas_modelo

,adilshamim8,samithsachidanandan,saurabhbadole
fact_salarios,71913,44244,46637
dim_tiempo,6,6,6
dim_experiencia,4,4,4
dim_empleo,4,4,4
dim_puesto,422,312,317
dim_pais,106,99,99
dim_tamano_empresa,3,3,3
dim_moneda,26,26,26
dim_modalidad,3,3,3


In [10]:
# ── Columnas, tipos y llave primaria de cada tabla (diseño físico) ─────────
# El diseño es idéntico en los tres datasets; se muestra adilshamim8.
esquema_fisico = pd.concat([M.esquema_tabla(con, t) for t in tablas_del_modelo('adilshamim8')], ignore_index=True)
esquema_fisico

,tabla,column_name,column_type,null,key
0,fact_salarios_adilshamim8,salario_id,INTEGER,NO,PRI
1,fact_salarios_adilshamim8,anio,SMALLINT,NO,None
2,fact_salarios_adilshamim8,experiencia_id,INTEGER,NO,None
3,fact_salarios_adilshamim8,empleo_id,INTEGER,NO,None
4,fact_salarios_adilshamim8,puesto_id,INTEGER,NO,None
5,fact_salarios_adilshamim8,residencia_pais_id,INTEGER,NO,None
6,fact_salarios_adilshamim8,empresa_pais_id,INTEGER,NO,None
7,fact_salarios_adilshamim8,tamano_id,INTEGER,NO,None
8,fact_salarios_adilshamim8,moneda_id,INTEGER,NO,None
9,fact_salarios_adilshamim8,remote_ratio,TINYINT,NO,None


In [11]:
# ── Ver algunas tablas del esquema estrella ────────────────────────────────
for t in ['dim_experiencia_adilshamim8', 'dim_empleo_adilshamim8', 'dim_tamano_empresa_adilshamim8',
          'dim_modalidad_adilshamim8', 'dim_tiempo_adilshamim8']:
    print(t); display(con.sql(f'SELECT * FROM {t} ORDER BY 1').df())

con.sql('SELECT * FROM fact_salarios_adilshamim8 LIMIT 5').df()

dim_experiencia_adilshamim8


,experiencia_id,codigo,nivel,orden
0,1,EN,Inicial (Entry/Junior),1
1,2,MI,Intermedio (Mid),2
2,3,SE,Senior,3
3,4,EX,Ejecutivo / Director,4


dim_empleo_adilshamim8


,empleo_id,codigo,tipo
0,1,CT,Contrato
1,2,FL,Freelance
2,3,FT,Tiempo completo
3,4,PT,Medio tiempo


dim_tamano_empresa_adilshamim8


,tamano_id,codigo,tamano
0,1,S,Pequeña (< 50 empleados)
1,2,M,Mediana (50 a 250)
2,3,L,Grande (> 250)


dim_modalidad_adilshamim8


,remote_ratio,modalidad,es_remoto_total
0,0,Presencial,False
1,50,Híbrido,False
2,100,Remoto total,True


dim_tiempo_adilshamim8


,anio,decada,es_anio_mas_reciente
0,2020,2020,False
1,2021,2020,False
2,2022,2020,False
3,2023,2020,False
4,2024,2020,False
5,2025,2020,True


,salario_id,anio,experiencia_id,empleo_id,puesto_id,residencia_pais_id,empresa_pais_id,tamano_id,moneda_id,remote_ratio,salario,salario_usd,tasa_cambio_a_usd,misma_ubicacion,es_atipico,_row_hash,_batch_id
0,1,2025,3,3,246,100,100,2,25,100,135000,135000,1.0,True,False,0001ea5db8e1dedd,1
1,2,2024,2,3,246,100,100,2,25,0,162240,162240,1.0,True,False,0002039698e12798,1
2,3,2025,1,3,128,100,100,2,25,0,114911,114911,1.0,True,False,00061938aff92dce,1
3,4,2024,2,3,246,100,100,2,25,0,84600,84600,1.0,True,False,0008a09cca20224d,1
4,5,2024,3,3,391,100,100,2,25,100,125000,125000,1.0,True,False,00097d288f80296e,1


---
## PASO 4 — Validar el modelo

Al repartir los datos en tablas no se debe perder ni duplicar nada: **mismas filas**, **misma suma de la medida principal** (`salary_in_usd` → `salario_usd`) y **ninguna fila de hechos sin dimensión (huérfana)**.

In [12]:
validacion = []
for et in tools.ETIQUETAS:
    v = M.validar(con, f'base_{et}', f'fact_salarios_{et}', 'salary_in_usd', 'salario_usd')
    h = M.huerfanas(con, f'fact_salarios_{et}', relaciones(et))
    v.insert(0, 'dataset', et)
    v['filas_huerfanas'] = int(h['filas_huerfanas'].iloc[0])
    validacion.append(v)

validacion = pd.concat(validacion, ignore_index=True)
validacion

,dataset,filas_tabla_plana,filas_hechos,suma_tabla_plana,suma_hechos,cuadra,filas_huerfanas
0,adilshamim8,71913,71913,10870459160,10870459160,True,0
1,samithsachidanandan,44244,44244,6694711653,6694711653,True,0
2,saurabhbadole,46637,46637,7061393049,7061393049,True,0


In [13]:
assert validacion['cuadra'].all(), 'Las filas o la suma no cuadran'
assert (validacion['filas_huerfanas'] == 0).all(), 'Hay filas huérfanas'
print('Validación correcta: mismas filas, misma suma de salary_in_usd y 0 huérfanas en los 3 modelos.')

Validación correcta: mismas filas, misma suma de salary_in_usd y 0 huérfanas en los 3 modelos.


---
## PASO 5 — Consultar el modelo

Toda consulta sobre una estrella tiene la misma forma: los hechos al centro, unidos solo a las dimensiones que pide la pregunta.

In [14]:
# ── ¿Cuánto gana cada nivel de experiencia en el año más reciente? ─────────
def salario_por_nivel(et):
    return con.sql(f"""
        SELECT x.nivel,
               COUNT(*)                          AS registros,
               ROUND(AVG(f.salario_usd))         AS promedio_usd,
               ROUND(MEDIAN(f.salario_usd))      AS mediana_usd
        FROM fact_salarios_{et}      AS f
        JOIN dim_experiencia_{et}    AS x USING (experiencia_id)
        JOIN dim_tiempo_{et}         AS t USING (anio)
        WHERE t.es_anio_mas_reciente
        GROUP BY x.nivel, x.orden
        ORDER BY x.orden
    """).df()

for et in tools.ETIQUETAS:
    print(et); display(salario_por_nivel(et))

adilshamim8


,nivel,registros,promedio_usd,mediana_usd
0,Inicial (Entry/Junior),4057,97404.0,85000.0
1,Intermedio (Mid),12575,135420.0,120900.0
2,Senior,19865,167935.0,155567.0
3,Ejecutivo / Director,1785,203500.0,190000.0


samithsachidanandan


,nivel,registros,promedio_usd,mediana_usd
0,Inicial (Entry/Junior),1167,93842.0,84100.0
1,Intermedio (Mid),3504,138730.0,124400.0
2,Senior,5640,171001.0,158100.0
3,Ejecutivo / Director,311,199976.0,181000.0


saurabhbadole


,nivel,registros,promedio_usd,mediana_usd
0,Inicial (Entry/Junior),1476,95017.0,84525.0
1,Intermedio (Mid),4230,139249.0,124942.0
2,Senior,6914,170678.0,158000.0
3,Ejecutivo / Director,393,201951.0,183544.0


In [15]:
# ── ¿Pagan más por trabajar en remoto? Mediana por modalidad y tamaño de empresa ──
con.sql("""
    SELECT mo.modalidad,
           te.tamano,
           COUNT(*)                       AS registros,
           ROUND(MEDIAN(f.salario_usd))   AS mediana_usd
    FROM fact_salarios_adilshamim8        AS f
    JOIN dim_modalidad_adilshamim8        AS mo USING (remote_ratio)
    JOIN dim_tamano_empresa_adilshamim8   AS te USING (tamano_id)
    GROUP BY mo.modalidad, te.tamano
    ORDER BY mo.modalidad, mediana_usd DESC
""").df()

,modalidad,tamano,registros,mediana_usd
0,Híbrido,Grande (> 250),213,74130.0
1,Híbrido,Pequeña (< 50 empleados),44,72000.0
2,Híbrido,Mediana (50 a 250),70,59551.0
3,Presencial,Grande (> 250),1021,143200.0
4,Presencial,Mediana (50 a 250),53007,138230.0
5,Presencial,Pequeña (< 50 empleados),53,100000.0
6,Remoto total,Mediana (50 a 250),17033,141000.0
7,Remoto total,Grande (> 250),354,110000.0
8,Remoto total,Pequeña (< 50 empleados),118,69017.0


In [16]:
# ── Uso de la dimensión de rol doble: empleados que viven en un país distinto al de la empresa ──
con.sql("""
    SELECT pe.codigo_iso2                AS pais_empresa,
           COUNT(*)                      AS empleados_en_otro_pais,
           ROUND(MEDIAN(f.salario_usd))  AS mediana_usd
    FROM fact_salarios_adilshamim8 AS f
    JOIN dim_pais_adilshamim8      AS pr ON pr.pais_id = f.residencia_pais_id
    JOIN dim_pais_adilshamim8      AS pe ON pe.pais_id = f.empresa_pais_id
    WHERE pr.pais_id <> pe.pais_id
    GROUP BY pe.codigo_iso2
    ORDER BY empleados_en_otro_pais DESC
    LIMIT 10
""").df()

,pais_empresa,empleados_en_otro_pais,mediana_usd
0,US,69,70000.0
1,DE,15,55475.0
2,GB,10,71428.0
3,AU,7,40000.0
4,CA,6,54111.0
5,FR,4,117223.0
6,AS,4,38500.0
7,PL,3,54000.0
8,RU,3,57000.0
9,NL,2,61356.0


In [17]:
# ── Cerrar la conexión ─────────────────────────────────────────────────────
con.execute('CHECKPOINT')
con.close()
print(f'Modelo guardado en: {tools.SERVING_DB.relative_to(tools.BASE_DIR)}')

M.ver_serving()

Modelo guardado en: Modelado_de_Datos/serving.db

  DuckDB: serving.db

  NOMBRE                               TIPO        FILAS  COLS
  ------------------------------------ ------ ---------- -----
  dim_empleo_adilshamim8               tabla           4     3
  dim_empleo_samithsachidanandan       tabla           4     3
  dim_empleo_saurabhbadole             tabla           4     3
  dim_experiencia_adilshamim8          tabla           4     4
  dim_experiencia_samithsachidanandan  tabla           4     4
  dim_experiencia_saurabhbadole        tabla           4     4
  dim_modalidad_adilshamim8            tabla           3     3
  dim_modalidad_samithsachidanandan    tabla           3     3
  dim_modalidad_saurabhbadole          tabla           3     3
  dim_moneda_adilshamim8               tabla          26     3
  dim_moneda_samithsachidanandan       tabla          26     3
  dim_moneda_saurabhbadole             tabla          26     3
  dim_pais_adilshamim8                 tabla  

## Resumen

| Paso | Tema | Resultado |
|---|---|---|
| 1 | — | Vista temporal `base_<etiqueta>` sobre cada Parquet |
| 2 | 3.1 Lógico | Qué columnas son dimensiones, cuáles hechos y cuáles se descartan |
| 3 | 3.1 Físico + 3.2 | 3 esquemas estrella en `serving.db`: 8 dimensiones + `fact_salarios_<etiqueta>` cada uno (27 tablas, ningún nombre repetido) |
| 4 | — | Mismas filas, misma suma de `salary_in_usd`, 0 huérfanas |
| 5 | — | Consultas con hechos + dimensiones |